<a href="https://colab.research.google.com/github/Arwaabulails/Rocket-CodeCamp/blob/main/model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U tensorflow


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
from sklearn.model_selection import KFold, train_test_split
from tensorflow.keras import mixed_precision
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi


In [ ]:
img_path='/content/drive/MyDrive/red spider final'


In [ ]:
print(img_path)
import os
labels=[]
for i in os.listdir(img_path):
  if os.path.isdir(os.path.join(img_path,i)):
    labels.append(i)
print(labels)

In [ ]:
'''from tensorflow.keras.utils import image_dataset_from_directory # Import the required function

# Load dataset and split into train, validation, and test sets
train_ds = image_dataset_from_directory(
    img_path,
    validation_split=0.3,  # 20% of data will be used for validation/testing
    subset="training",
    seed=123,
    image_size=(224, 224),  # Resizing images for MobileNetV2

)'''

In [ ]:
'''val_test_ds = tf.keras.utils.image_dataset_from_directory(
    img_path,
    validation_split=0.3,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=BATCHSIZE
)'''

In [ ]:
'''
val_size = int(0.5 * tf.data.experimental.cardinality(val_test_ds).numpy())  # ~132 images
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

# Verify sizes (approximate due to batching)
print(f"Train size: {tf.data.experimental.cardinality(train_ds).numpy() * BATCHSIZE}")
print(f"Val size: {tf.data.experimental.cardinality(val_ds).numpy() * BATCHSIZE}")
print(f"Test size: {tf.data.experimental.cardinality(test_ds).numpy() * BATCHSIZE}")
'''

In [ ]:
data_dir = "/content/drive/MyDrive/red spider final"

In [ ]:
'''
# Load dataset
full_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    seed=123,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCHSIZE
)

# Unbatch and get exact count
images_labels = list(full_ds.unbatch())
total_images = len(images_labels)  # 879
print(f"Total images: {total_images}")

# Separate images and labels
images = [x[0] for x in images_labels]
labels = [x[1] for x in images_labels]

# Exact split
train_size = int(0.7 * total_images)  # 615
val_size = int(0.15 * total_images)   # 131
test_size = total_images - train_size - val_size  # 133

# Create datasets without padding
train_ds = tf.data.Dataset.from_tensor_slices(
    (images[:train_size], labels[:train_size])
).batch(BATCHSIZE, drop_remainder=True)  # Drop partial batches

val_ds = tf.data.Dataset.from_tensor_slices(
    (images[train_size:train_size + val_size], labels[train_size:train_size + val_size])
).batch(BATCHSIZE, drop_remainder=True)

test_ds = tf.data.Dataset.from_tensor_slices(
    (images[train_size + val_size:], labels[train_size + val_size:])
).batch(BATCHSIZE, drop_remainder=True)


# Unbatch and get exact count
images_labels = list(full_ds.unbatch())
total_images = len(images_labels)  # 879
print(f"Total images: {total_images}")

# Separate images and labels
images = [x[0] for x in images_labels]
labels = [x[1] for x in images_labels]

# Exact split
train_size = int(0.7 * total_images)  # 615
val_size = int(0.15 * total_images)   # 131
test_size = total_images - train_size - val_size  # 133

# Create datasets without padding
train_ds = tf.data.Dataset.from_tensor_slices(
    (images[:train_size], labels[:train_size])
).batch(BATCHSIZE, drop_remainder=True)  # Drop partial batches

val_ds = tf.data.Dataset.from_tensor_slices(
    (images[train_size:train_size + val_size], labels[train_size:train_size + val_size])
).batch(BATCHSIZE, drop_remainder=True)

test_ds = tf.data.Dataset.from_tensor_slices(
    (images[train_size + val_size:], labels[train_size + val_size:])
).batch(BATCHSIZE, drop_remainder=True)

'''

In [ ]:
'''
# Print sizes
print(f"Train size: {tf.data.experimental.cardinality(train_ds).numpy() * BATCHSIZE}")
print(f"Val size: {tf.data.experimental.cardinality(val_ds).numpy() *BATCHSIZE}")
print(f"Test size: {tf.data.experimental.cardinality(test_ds).numpy() * BATCHSIZE}")

# Verify actual counts
print(f"Actual Train: {len(list(train_ds.unbatch()))}")
print(f"Actual Val: {len(list(val_ds.unbatch()))}")
print(f"Actual Test: {len(list(test_ds.unbatch()))}")

'''

In [ ]:
BATCHSIZE=32
IMG_SIZE=224
EPOCHS=100

In [ ]:
import tensorflow as tf
import numpy as np
from collections import defaultdict
import random

# Define constants
data_dir = "/content/drive/MyDrive/red spider final"  # Replace with the actual path to your dataset
IMG_SIZE = 224  # Image size (e.g., 224x224)
BATCH_SIZE = BATCHSIZE # Batch size
TRAIN_SPLIT = 0.7  # 70% for training
VAL_SPLIT = 0.15  # 15% for validation
SEED = 123  # Random seed for reproducibility

 #Load dataset
full_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None  # Load unbatched for easier manipulation
)

In [ ]:


# Get images and labels
images_labels = list(full_ds)  # List of (image, label) tuples
total_images = len(images_labels)
print(f"Total images: {total_images}")

# Group images by class
class_images = defaultdict(list)
for img, lbl in images_labels:
    # Convert label to Python scalar
    lbl_scalar = int(lbl.numpy()) if isinstance(lbl, tf.Tensor) else int(lbl)
    class_images[lbl_scalar].append((img, lbl))

# Stratified split
train_data = []
val_data = []
test_data = []

for class_id, samples in class_images.items():
    # Shuffle samples for this class
    random.seed(SEED)
    random.shuffle(samples)

    # Calculate split sizes for this class
    n_samples = len(samples)
    n_train = int(TRAIN_SPLIT * n_samples)
    n_val = int(VAL_SPLIT * n_samples)
    n_test = n_samples - n_train - n_val

    # Assign samples to splits
    train_data.extend(samples[:n_train])
    val_data.extend(samples[n_train:n_train + n_val])
    test_data.extend(samples[n_train + n_val:])

# Shuffle the splits
random.seed(SEED)
random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    ([x[0] for x in train_data], [x[1] for x in train_data])
).batch(BATCH_SIZE, drop_remainder=True).cache().shuffle(buffer_size=len(train_data)).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    ([x[0] for x in val_data], [x[1] for x in val_data])
).batch(BATCH_SIZE, drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    ([x[0] for x in test_data], [x[1] for x in test_data])
).batch(BATCH_SIZE, drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)

# Verify the splits
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# Verify class distribution
def print_class_distribution(data, name):
    labels = [int(x[1].numpy()) if isinstance(x[1], tf.Tensor) else int(x[1]) for x in data]
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n{name} class distribution:")
    for cls, count in zip(unique, counts):
        print(f"Class {cls}: {count} samples ({count/len(labels)*100:.2f}%)")

print_class_distribution(train_data, "Training")
print_class_distribution(val_data, "Validation")
print_class_distribution(test_data, "Test")

In [ ]:
#visualization
class_names = os.listdir(data_dir)

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(5):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        # Convert label to Python scalar for indexing
        label_index = int(labels[i].numpy()) if isinstance(labels[i], tf.Tensor) else int(labels[i])
        plt.title(class_names[label_index])  # Use class_names to get the label

        plt.axis("off")

In [ ]:
for image_batch, labels_batch in train_ds:
    print(image_batch.shape)
    print(labels_batch.shape)
    break

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
normalizations_layer = layers.Rescaling(1./255)

In [ ]:
normalized_ds=train_ds.map(lambda x,y:(normalizations_layer(x),y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

In [ ]:

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal', input_shape=(IMG_SIZE, IMG_SIZE, 3)),  # Random horizontal flip
    layers.RandomRotation(0.15),  # Random rotation (-20% to +20%)
   # layers.RandomBrightness(factor=(0.7, 1.3)),  # Adjust brightness in the range 0.7–1.3
    layers.RandomZoom(0.15)  # Random zoom by 5%
])

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),

                                            include_top=False, weights='imagenet')

base_model.trainable = False
#for layer in base_model.layers[:120]:
    #layer.trainable = False


In [ ]:
#base_model.summary()

In [ ]:
#setupe the model architecture
l2_lambda =0.0005
inputs=keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
X=data_augmentation(inputs)
x=keras.applications.mobilenet_v2.preprocess_input(X)
x=base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.2)(x)
x=layers.Dense(256,kernel_regularizer=tf.keras.regularizers.l2(l2_lambda))(x)
x=layers.BatchNormalization()(x)
x = layers.ReLU()(x)
#x=layers.Dropout(0.5)(x)   #0.5###########
x=layers.Dense(128,kernel_regularizer=tf.keras.regularizers.l2(l2_lambda))(x)
x=layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x=layers.Dropout(0.5)(x)   #0.5


#output layar
# The prediction layer should match the output of the preceding layer.
prediction_layer = tf.keras.layers.Dense(4, activation="softmax")
outputs=prediction_layer(x)
model=keras.Model(inputs,outputs)

In [ ]:
model.summary()

In [ ]:
callbacks=[
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5

    )

]

In [ ]:
import numpy as np

# Convert labels to NumPy array before concatenation
labels = np.array([y.numpy() for _, y in train_ds.unbatch()])

# Now you can use bincount
print(np.bincount(labels))  # بيطبع عدد الصور لكل فئة


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


# Ensure 'classes' is a NumPy array
class_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2, 3]), y=labels)
class_weights_dict = dict(enumerate(class_weights))

In [ ]:
lr=1e-4

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    class_weight=class_weights_dict

)

In [ ]:
score = model.evaluate(test_ds)

In [ ]:
model.save('red_spider.h5')

In [ ]:
#convert the model into tflite

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('red_spider_91.35.tflite', 'wb') as f:
    f.write(tflite_model)


In [ ]:
# Get the actual number of epochs the training ran for
num_epochs = len(history.history['accuracy'])

# Adjust the epochs_range accordingly
epochs_range = range(num_epochs)

# Now plot the data
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['accuracy'], label='Training Accuracy') # Access 'accuracy' from history
plt.plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy') # Access 'val_accuracy' from history
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Training Loss') # Access 'loss' from history
plt.plot(epochs_range, history.history['val_loss'], label='Validation Loss') # Access 'val_loss' from history
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(images[i].numpy())
    img_array = tf.expand_dims(img_array, 0)
    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * (np.max(predictions[0])), 2)
    return predicted_class, confidence

In [ ]:
plt.figure(figsize=(15, 15))
for images, labels in test_ds.take(5):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        predicted_class, confidence = predict(model, images[i].numpy())
        actual_class = class_names[labels[i]]
        plt.title(f"Actual: {actual_class},\n Predicted: {predicted_class}.\n Confidence: {confidence}%")
        plt.axis("off")

In [ ]:


import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# Assuming 'model' and 'test_ds' are defined from the previous code

# Get predictions on the test set
y_pred = model.predict(test_ds)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = np.concatenate([y for x, y in test_ds], axis=0)

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

# Add labels to each cell
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, cm[i, j],
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


In [ ]:
# prompt: SHOW me the 9 images from moderate that the model make mistake in it and predict it mild and i need 9 image in one display with the name of images in the dataset

import matplotlib.pyplot as plt
import numpy as np

# Assuming 'model', 'test_ds', and 'class_names' are defined from the previous code

# Function to predict and get confidence
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * (np.max(predictions[0])), 2)
    return predicted_class, confidence

# Find misclassified images
misclassified_moderate_as_mild = []
for images, labels in test_ds:
    for i in range(images.shape[0]):
        image = images[i].numpy().astype("uint8")
        true_label = class_names[labels[i]]
        predicted_class, _ = predict(model, images[i].numpy())

        if true_label == "moderate" and predicted_class == "mild":
          misclassified_moderate_as_mild.append((image, true_label, predicted_class))


# Display the first 9 misclassified images
plt.figure(figsize=(15, 15))
for i in range(min(9, len(misclassified_moderate_as_mild))):
    image, true_label, predicted_class = misclassified_moderate_as_mild[i]
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(f"Actual: {true_label}\nPredicted: {predicted_class}")
    plt.axis("off")

plt.show()


In [ ]:
# print Precision, Recall, F1-score
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred_classes))


In [ ]:
base_model.trainable = True
# تجميد الطبقات الابتدائية فقط (مثل أول 100 طبقة)
for layer in base_model.layers[:100]:
    layer.trainable = False

# إعادة تجميع النموذج بمعدل تعلم منخفض
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# إعادة التدريب
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,


)

In [ ]:
# Let's take a look to see how many layers are in the base model
print("Number of layers in the base model: ", len(base_model.layers))
# Fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all the layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False